# 04 — Training and Forecasting

**Purpose:** load the fitted model artifacts, generate current lifecycle forecasts, build the 14-day → 7-day holdout comparison, and export stakeholder-facing outputs. Training logic itself remains in the modeling module.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\BaneleS\Downloads\Game Performance Model


In [2]:
import pandas as pd

from game_performance.config import SETTINGS
from game_performance.io import load_artifact, load_dataframe, save_dataframe
from game_performance.reporting import build_holdout_comparison, export_excel
from game_performance.scoring import run_daily_lifecycle_scoring

SETTINGS.ensure_directories()

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")
training_results = load_artifact(SETTINGS.model_dir / "training_results.joblib")
model_bundles = training_results["model_bundles"]

print("Loaded cleaned data:", df.shape)
print("Loaded model families:", len(model_bundles))


Loaded cleaned data: (880049, 13)
Loaded model families: 1


In [3]:
current_predictions, scoring_log = run_daily_lifecycle_scoring(
    df,
    model_bundles,
    SETTINGS.horizon_strategy,
    as_of_date=df["Date"].max(),
    exact_checkpoint_only=False,
)

print(f"Generated {len(current_predictions):,} predictions")
display(current_predictions.head(30))

if scoring_log is not None and len(scoring_log): 
    print("Scoring log")
    display(scoring_log)


Generated 12,491 predictions


,Game_ID,TournamentName,SystemName,First_Observed_Date,Lifecycle_Stage,Obs_Window,Horizon,active_observation_days,observation_coverage,predicted_clients_week_4,confidence_clients,predicted_stakes_week_4,confidence_stakes,recommendation,scored_at,Checkpoint,As_Of_Date
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,AmusnetS-100 Bulky Fruits,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,461.667089,0.745828,2.064080e+05,0.653370,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
1,AmusnetSlots::AmusnetS-100 Cats,AmusnetS-100 Cats,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,9012.851609,0.663291,1.893258e+06,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
2,AmusnetSlots::AmusnetS-100 Power Hot,AmusnetS-100 Power Hot,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,9856.281560,0.663291,2.924534e+06,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
3,AmusnetSlots::AmusnetS-2 Dragons,AmusnetS-2 Dragons,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,802.892905,0.745828,1.001033e+05,0.653370,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,AmusnetS-20 Bulky Fruits,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,3237.765547,0.717380,7.103374e+05,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
5,AmusnetSlots::AmusnetS-20 Diamonds,AmusnetS-20 Diamonds,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,614.731158,0.745828,2.911287e+05,0.809151,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
6,AmusnetSlots::AmusnetS-20 Golden Coins,AmusnetS-20 Golden Coins,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,1029.695513,0.745828,2.960207e+05,0.809151,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
7,AmusnetSlots::AmusnetS-20 Hot Blast,AmusnetS-20 Hot Blast,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,6334.590420,0.663291,1.322623e+06,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
8,AmusnetSlots::AmusnetS-20 Joker Reels,AmusnetS-20 Joker Reels,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,5688.883183,0.663291,1.379643e+06,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29
9,AmusnetSlots::AmusnetS-20 Power Hot,AmusnetS-20 Power Hot,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,4334.989950,0.663291,9.361747e+05,0.680523,Priority placement for week 4,2026-08-27 09:36:55.418659+00:00,14,2026-07-29


Scoring log


,Game_ID,Checkpoint,Horizon,As_Of_Date,scored_at
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
1,AmusnetSlots::AmusnetS-100 Cats,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
2,AmusnetSlots::AmusnetS-100 Power Hot,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
3,AmusnetSlots::AmusnetS-2 Dragons,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
...,...,...,...,...,...
12486,YGG::YGG-Goals of Olympus BoltLock,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
12487,YGG::YGG-Lucky Coin Festival,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
12488,YGG::YGG-Sidewinder Xtra Power,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00
12489,YGG::YGG-The Good Farmer,14,14,2026-07-29,2026-08-27 09:36:55.418659+00:00


In [4]:
holdout = training_results["holdout_predictions"]

print("Holdout predictions:", holdout.shape)

if holdout.empty:
    raise ValueError(
        "No holdout predictions are available. Run 03_model_experiments_production.ipynb first."
    )

available = (
    holdout[["Obs_Window", "Horizon"]]
    .drop_duplicates()
    .sort_values(["Obs_Window", "Horizon"])
    .reset_index(drop=True)
)

print("Available holdout model families:")
display(available)

available_pairs = {
    (int(row.Obs_Window), int(row.Horizon))
    for row in available.itertuples(index=False)
}

# Production preference. If full production training was run, this should exist.
preferred_pair = (14, 7)

if preferred_pair in available_pairs:
    checkpoint, horizon = preferred_pair
else:
    # Keep the notebook runnable, but make the issue obvious.
    checkpoint, horizon = tuple(available.iloc[-1].astype(int))
    print(
        f"WARNING: preferred production pair {preferred_pair} is not available. "
        f"Using checkpoint={checkpoint}, horizon={horizon} instead. "
        "Check skipped_configurations from notebook 03."
    )

comparison = build_holdout_comparison(
    holdout,
    checkpoint=int(checkpoint),
    horizon=int(horizon),
)

print(f"Comparison rows: {len(comparison):,}")
display(comparison.head(20))


Holdout predictions: (498, 24)
Available holdout model families:


,Obs_Window,Horizon
0,14,14


Comparison rows: 498


,Game_ID,TournamentName,SystemName,First_Observed_Date,Observation_Start,Observation_End,Prediction_Start,Prediction_End,Obs_Window,Horizon,...,variance_clients,probability_clients,actual_stakes_week_3,actual_stakes_week_4,week_3_to_week_4_stakes_growth_pct,baseline_stakes_week_4,predicted_stakes_week_4,confidence_stakes,variance_stakes,probability_stakes
0,SyX-Online::Iceland League U20 Women,Iceland League U20 Women,SyX-Online,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,-4939.110023,0.047607,<NA>,71591.04,<NA>,9.176420e+03,4.093222e+03,0.033815,-67497.817529,0.057175
1,Colossus::COL-WC,COL-WC,Colossus,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,-116.204224,0.793964,16636.24,18361.95,0.103732,1.142406e+04,1.501692e+04,0.236707,-3345.027624,0.817828
2,Erlang-Online::Iceland League U20 Women,Iceland League U20 Women,Erlang-Online,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,<NA>
3,SplitThePot::STP-daredevil,STP-daredevil,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,483.395544,0.0,246394.53,105009.68,-0.573815,2.865507e+05,2.110142e+05,0.653370,106004.547072,0.0
4,SplitThePot::STP-paperplane,STP-paperplane,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,467.934057,0.0,654431.99,32577.46,-0.95022,5.940401e+04,1.182908e+05,0.653370,85713.351663,0.0
5,SplitThePot::STP-Rugby Run Twins,STP-Rugby Run Twins,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,1345.9148,0.0,1433806.98,577266.48,-0.597389,5.663672e+05,5.660376e+05,0.648555,-11228.844999,0.980548
6,SplitThePot::STP-Zama Fruits,STP-Zama Fruits,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,8154.176538,0.126588,6595433.69,2434423.63,-0.630893,4.611966e+06,3.475569e+06,0.680523,1041145.297086,0.572324
7,SplitThePot::STP-Zama Fruits-Data Free,STP-Zama Fruits-Data Free,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,3965.098066,0.41664,2354337.11,900473.73,-0.617526,1.688709e+06,1.022238e+06,0.680523,121764.7616,0.864777
8,SplitThePot Free Spins::STP-Zama Fruits,STP-Zama Fruits,SplitThePot Free Spins,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,39.820553,0.0,0.0,0.0,<NA>,0.000000e+00,7.425257e+04,0.653370,74252.570969,0.0
9,SplitThePot Free Spins::STP-Zama Fruits-Data Free,STP-Zama Fruits-Data Free,SplitThePot Free Spins,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,214.073824,0.0,0.0,0.0,<NA>,0.000000e+00,7.419361e+04,0.653370,74193.611199,0.0


In [5]:
from datetime import datetime
from pathlib import Path


def export_production_excel(
    path: Path,
    *,
    comparison,
    forecasts,
    regression,
    classification,
) -> Path:
    """Export the production report, using a timestamped copy if Excel has it open."""
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(path, "a"):
            pass
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")
        print("Report is currently open.\n" f"Saving instead to:\n{path}")

    return export_excel(
        path,
        comparison=comparison,
        forecasts=forecasts,
        regression=regression,
        classification=classification,
    )


In [6]:
save_dataframe(current_predictions, SETTINGS.report_dir / "current_predictions.pkl")

report_path = SETTINGS.report_dir / "model_business_report.xlsx"

export_production_excel(
    report_path,
    comparison=comparison,
    forecasts=current_predictions,
    regression=training_results["regression_evaluation"],
    classification=training_results["classification_evaluation"],
)

print("Saved current predictions:", SETTINGS.report_dir / "current_predictions.pkl")
print("Saved Excel report:", report_path)


Saved current predictions: C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\reports\current_predictions.pkl
Saved Excel report: C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\reports\model_business_report.xlsx
